# Feature interpretation for CFU prediction model

## Load data

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
root = Path.cwd().parent
sys.path.insert(0, str(root))

Load TPM data and CFU counts.

In [ ]:
from src.tpm_data import get_current_data
from src.eda import get_annotations

data_df = get_current_data(root)
annotations = get_annotations(root)

## Feature interpretation

Get model.

In [ ]:
# Get path to models
model_path = root / "models" / "diagonal_cfu_model.pkl"
forward_model = joblib.load(model_path)

Plot coefficient distribution.

In [ ]:
coefs = forward_model.named_steps["model"].coef_.ravel()

plt.hist(coefs, bins = 20)
plt.xlabel("PLS regression coefficient")
plt.ylabel("Frequency")
plt.title("Hisogram of regression coefficients")

Rank features by coefficient.

In [ ]:
from src.interpret import plot_top_features

# Store coefficients in df
coef_df = pd.DataFrame(
    {"coef": coefs}, 
    index = data_df.columns[data_df.columns.str.contains("SP")]
)

# Plot top features
plot_top_features(
    coef_df = coef_df,
    annot_df = annotations,
    top_n = 30,
    xlabel = "Regression coefficient",
    title = "Top 30 features for PLS regression model"   
)

Run GSEA on ranked features.

In [ ]:
from src.interpret import run_custom_gsea

gs = run_custom_gsea(
    coef_df = coef_df,
    annot_df = annotations,
    set_col = "GENE.CATEGORY",
    seed = 111
)

Enrichment plots for top 3 pathways.

In [ ]:
terms = gs.res2d.Term
ax1 = gs.plot(terms = terms[0])
ax2 = gs.plot(terms = terms[1])
ax3 = gs.plot(terms = terms[2])


Dotplot of top pathway hits.

In [ ]:
from gseapy import dotplot

ax = dotplot(
    gs.res2d,
    column = "FDR q-val",
    cutoff = 0.25
)